In [ ]:
# @title Config

!pip install cartopy
!pip install torch_geometric
!pip install torchcde

from google.colab import drive

import pickle
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import argparse
import yaml
import torch
import os
import argparse
import yaml
from pathlib import Path
import json
import sys
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import torch.nn.functional as F
from typing import Any, Callable, Tuple, Union
from torch import nn, optim
from torch.optim import Adam
import networkx as nx
import random
from torch.autograd import Variable
import gc
import torchcde
import math
from statsmodels.tsa.seasonal import STL
import matplotlib.pyplot as plt

import numpy as np
import torch
from torch.optim import Adam
from tqdm import tqdm
import pickle
import logging
import time
import xarray as xr

drive.mount('/content/drive', force_remount=True)
os.chdir('/content/drive/My Drive/CLCRN-main')

## Specify configuration for NLDAS data
dlwfm = 'CLCRN'
dataset = 'NLDAS'
dataName = 'tmp2m'
baseline = 'fable'                                                                # ['fgsm', 'taaowpf', 'noise_attack', 'stpgd', 'ala', ''fable']

stepNum = 1000
seq_length = 12
num_locs = 10
num_runs = 5

lr = 0.1
test_sample_num = 0
use_optimizer = 'Adam' # ['Adam', 'PGD', 'pure']

reg = 1
target_key = ['LLH', 'LHL', 'LHH', 'HLL', 'HLH', 'HHL', 'HHH']                     # For 3d transform, ['LLL', 'LLH', 'LHL', 'LHH', 'HLL', 'HLH', 'HHL', 'HHH']
if reg == 1:
    coeff_weights = {'LLH': 0.8, 'LHL': 0.8, 'HLL': 0.8, 'LHH': 0.5, 'HLH': 0.5, 'HHL': 0.5, 'HHH': 0.2}
    reg_p = 2
    lambda_reg = 0.0001
clampEpsilon = 2.5 # [0.2, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0]

if dataset == 'ERA5':
  grid_size = 2048
elif dataset == 'NLDAS':
  grid_size = 1320

if dataset == 'ERA5' and dataName == 'T2M':
    mean = 278.272156
    std = 21.054403
    max_std = (317.822540 - mean) / std
    min_std = (193.666763 - mean) / std
    p99 = (302.390472 - mean) / std
    p95 = (300.539886 - mean) / std
    p90 = (299.632111 - mean) / std
    p75 = (295.900238 - mean) / std
    p50 = (283.214203 - mean)/std
    p25 = (268.810059 - mean) / std
elif dataset == 'ERA5' and dataName == 'TISR':
    mean = 6440561.000000
    std = 7721753.000000
    max_std = (27871204.000000 - mean) / std
    min_std = (0.000000 - mean) / std
    p99 = (25913028.000000 - mean) / std
    p95 = (22899602.000000 - mean) / std
    p90 = (19387050.000000 - mean) / std
    p75 = (11292315.000000 - mean) / std
    p50 = (2825003.250000 - mean)/std
    p25 = (0.000000 - mean) / std
elif dataset == 'NLDAS' and dataName == 'pressfc':
    mean = 9.393699e+04
    std = 9.866053e+03
    max_std = (104909.984375 - mean) / std
    min_std = (68400.7265625 - mean) / std
    p99 = (102205.46875 - mean) / std
    p95 = (101510.4609375 - mean) / std
    p90 = (100891.4453125 - mean) / std
    p75 = (9.907731e+04 - mean) / std
    p50 = (9.665904e+04 - mean)/std
    p25 = (8.935441e+04 - mean) / std
elif dataset == 'NLDAS' and dataName == 'tmp2m':
    mean = 2.839279e+02
    std = 3.833667e+01
    max_std = (3.145732e+02 - mean) / std
    min_std = (2.325919e+02 - mean) / std
    p99 = (304.43316650390625 - mean) / std
    p95 = (301.278076171875 - mean) / std
    p90 = (298.9938659667969 - mean) / std
    p75 = (2.934141e+02 - mean) / std
    p50 = (2.853937e+02 - mean)/std
    p25 = (2.754702e+02 - mean) / std
elif dataset == 'NLDAS' and dataName == 'apcpsfc':
    mean = 0.06717896
    std = 0.1449845
    max_std = (2.41478 - mean) / std
    p99 = (0.71667926 - mean) / std
    p95 = (0.36164382 - mean) / std
    p90 = (0.21418104 - mean) / std
    p75 = (0.06022248 - mean) / std
    p50 = (0.005008133 - mean)/std
    p25 = (0 - mean) / std
    min_std = (0 - mean) / std

if dataset == 'ERA5':
    batch_size = 30
elif dataset == 'NLDAS':
    batch_size = 48

## Specify configuration for adversatial target construction
forCaseStudy = 0                                                                # [0, 1]
noiseType = 'random_replace'                                                    # ['random_replace', 'case_study']
# tStart, tEnd = 0, 11 # 0, 11
# pStart, pEnd = 1000, 1013 # 0, 2048
selectedLocations = [1297,   # Gulf of Mexican: [1296, 1297, 1298, 1360, 1361, 1362]
                     1335,   # Janpan Ocean: [1335, 1336, 1337, 1399, 1400, 1401]
                     1195]   # Indian Ocean: [1194, 1195, 1196, 1258, 1259, 1260]

## Specify configuration for baseline methods
caseStudy = 0                                                                   # [0, 1]
locationForLoss = "all"                                                         # ['target', 'all']
findType = 'saliency'                                                           # ['random', 'saliency']
if dataset == 'ERA5':
    k = 1536                                                                         # the number of salient nodes we are looking for
elif dataset == 'NLDAS':
    k = 990

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

data_path = f"/content/drive/My Drive/{dataset}_{dataName}"
test_x_path = os.path.join(data_path, "test_X.pt")
test_y_path = os.path.join(data_path, "test_Y.pt")
save_dir_base = os.path.join(data_path, dlwfm, "original_forecasts_and_adversarial_targets")

# Read (lon, lat) for datasets
if dataset == 'ERA5':
    lat_lon_path = f"{data_path}/2m_temperature_2005_5.625deg.nc"
    ds = xr.open_dataset(lat_lon_path)
    latitudes = ds['lat'].values
    longitudes = ds['lon'].values
    lon_grid, lat_grid = np.meshgrid(longitudes, latitudes)
    lonlat = np.column_stack((lon_grid.ravel(), lat_grid.ravel()))
    lons = longitudes
    lats = latitudes
elif dataset == "NLDAS":
    lonlatAddr = f"{data_path}/position_info.pkl"
    with open(lonlatAddr, 'rb') as lonLatPkl:
        lonLatDict = pickle.load(lonLatPkl)
    lons = lonLatDict['lonlat'][:, 0]
    lats = lonLatDict['lonlat'][:, 1]
    lonlat = lonLatDict['lonlat']

save_dir_base = os.path.join(data_path, dlwfm, "original_forecasts_and_adversarial_targets")
os.makedirs(save_dir_base, exist_ok=True)
num_locs = 10
num_runs = 5

out_path_base = f"/content/drive/My Drive/{dataset}_{dataName}/{dlwfm}/{baseline}/attack_results"
eval_path_base = f"/content/drive/My Drive/{dataset}_{dataName}/{dlwfm}/{baseline}/eval_results"
os.makedirs(out_path_base, exist_ok=True)
os.makedirs(save_dir_base, exist_ok=True)
os.makedirs(eval_path_base, exist_ok=True)
os.makedirs(os.path.join(out_path_base, "attack_loss"), exist_ok=True)
os.makedirs(os.path.join(out_path_base, "loss_records"), exist_ok=True)

if dataset == 'ERA5':
    H = 32
    W = 64
elif dataset == 'NLDAS':
    H = 28
    W = 58
device = 'cuda'
visualize = 1


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 134.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 8.4 MB/s eta 0:00:00
Mounted at /content/drive


In [ ]:
# @title Import CLCRN Model

import sys
!{sys.executable} -m pip install ruamel.yaml
!{sys.executable} -m pip install torch-scatter -f https://data.pyg.org/whl/torch-$(python -c "import torch;print(torch.__version__)").html

from supervisor import Supervisor
import yaml
import torch_scatter

filePath = f"{data_path}/{dlwfm}/config_clcrn_{dataName}.yaml"
with open(filePath, 'r') as f:
    clcrnConfig = yaml.load(f, Loader=yaml.SafeLoader)

best_CLCRN_model_epoch_dict = {'tmp2m': 84,
                               'pressfc': 89,
                               'apcpsfc': 78}
best_CLCRN_model_epoch = best_CLCRN_model_epoch_dict[dataName]
print(f"best_CLCRN_model_epoch for {dataName} is {best_CLCRN_model_epoch}")

def readYaml(file_path):
    with open(file_path, 'r') as file:
        data = yaml.safe_load(file)
    return data

def recursiveReplace(data, old, new):
    if isinstance(data, dict):
        for key, value in data.items():
            data[key] = recursiveReplace(value, old, new)
    elif isinstance(data, list):
        data = [recursiveReplace(item, old, new) for item in data]
    elif isinstance(data, str):
        data = data.replace(old, new)
    return data

def writeYaml(data, file_path):
    with open(file_path, 'w') as file:
        yaml.safe_dump(data, file, default_flow_style=False, sort_keys=False)

data = readYaml(filePath)
kernelSize = (1,1)

clcrnModelTrained = Supervisor(**clcrnConfig)
clcrnModelTrained.load_model(best_CLCRN_model_epoch)
model = clcrnModelTrained.model
model.eval()


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 12.4 MB/s eta 0:00:00
Looking in links: https://data.pyg.org/whl/torch-2.10.0+cu128.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.0/108.0 kB 9.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for torch-scatter: filename=torch_scatter-2.1.2-cp312-cp312-linux_x86_64.whl size=3877462 sha256=e84efdf0fd5019703ed2ece1b7058557b19112d0bb6709a31ba159883ffa65c1
  Stored in directory: /root/.cache/pip/wheels/84/20/50/44800723f57cd798630e77b3ec83bc80bd26a1e3dc3a672ef5
Successfully built torch-scatter
best_CLCRN_model_epoch for tmp2m is 84
2026-02-24 08:12:27,458 - INFO - Log directory: experiments/tmp2m/tmp2m


INFO:supervisor:Log directory: experiments/tmp2m/tmp2m


loading data of trn set...
loading data of val set...
loading data of test set...
2026-02-24 08:12:59,751 - INFO - Model created


/content/drive/My Drive/CLCRN-main/model/clcnn/seq2seq.py:20: UserWarning: torch.sparse.SparseTensor(indices, values, shape, *, device=) is deprecated.  Please use torch.sparse_coo_tensor(indices, values, shape, dtype=, device=). (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:654.)
  angle_ratio = torch.sparse.FloatTensor(
INFO:supervisor:Model created


model loaded from:  experiments/tmp2m/tmp2m/saved_model
2026-02-24 08:13:02,525 - INFO - Loaded model at 84


INFO:supervisor:Loaded model at 84


CLCRNModel(
  (feature_embedding): Linear(in_features=1, out_features=8, bias=True)
  (conv_ker): CLConv(
    (lcker): LocalConditionalKer(
      (network): ModuleList(
        (0): Linear(in_features=4, out_features=4, bias=True)
        (1): Linear(in_features=4, out_features=8, bias=True)
        (2): BatchNorm1d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (3): Linear(in_features=8, out_features=1, bias=True)
      )
    )
    (weight_att): Attention(
      (query_linear): Linear(in_features=2, out_features=8, bias=True)
      (key_linear): Linear(in_features=2, out_features=8, bias=True)
    )
  )
  (encoder_model): EncoderModel(
    (conv): CLConv(
      (lcker): LocalConditionalKer(
        (network): ModuleList(
          (0): Linear(in_features=4, out_features=4, bias=True)
          (1): Linear(in_features=4, out_features=8, bias=True)
          (2): BatchNorm1d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (3): Lin

In [ ]:
#@title Import Adversarial Attack Methods
!pip install pysal esda libpysal

import statsmodels.api as sm
from esda.moran import Moran
from libpysal.weights import DistanceBand

import os
import time
import math
import gc
import random
import pandas as pd
import torch
import torch.nn.functional as F
import torch.optim as optim
from torch.optim import Adam

!pip install PyWavelets

import math
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import gc
import yaml
import numpy as np
import matplotlib.pyplot as plt
import pywt
import pandas as pd
import pickle
import torch
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import MaxNLocator
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import xarray as xr
import os

torch.cuda.empty_cache()
gc.collect()

class NoiseAttack:
    def __init__(self, forward_func, selectedLocations):
        self.forward_func = forward_func
        self.selectedLocations = selectedLocations

    def rmse(self, thisOutputs, target):
        loss = F.mse_loss(thisOutputs, target)
        loss = torch.sqrt(loss)

        return loss

    def perturb(self, inputs, target, clampEpsilon, stepNum):
        self.forward_func.eval()
        device = inputs.device
        originalInputs = inputs.clone().to(device)

        perturbedInputsBest = torch.empty_like(originalInputs)
        lossBest = float('inf')
        loss_history = []

        for epoch in range(stepNum):
            perturbation = ((p75 - p25) / 6) * torch.randn(size=originalInputs.shape)
            perturbation = perturbation.to(device)
            thisPerturbedInputs = perturbation + originalInputs
            thisPerturbedInputs = torch.clamp(thisPerturbedInputs,
                                              torch.maximum(originalInputs - clampEpsilon, torch.tensor(min_std, device=device)),
                                              torch.minimum(originalInputs + clampEpsilon, torch.tensor(max_std, device=device)))

            with torch.no_grad():
                thisOutputs = self.forward_func(thisPerturbedInputs)
                loss = self.rmse(thisOutputs, target)
            print(f"Epoch {epoch}, Loss: {loss.item()}")
            loss_history.append(loss.item())

            if loss.item() < lossBest:
                lossBest = loss.item()
                perturbedInputsBest.copy_(thisPerturbedInputs)

            del thisOutputs
            del loss

        torch.cuda.empty_cache()
        gc.collect()

        return perturbedInputsBest, loss_history

class ALA:
    def __init__(self, forward_func, selectedLocations):
        self.forward_func = forward_func
        self.epsilon = 1e-8
        self.selectedLocations = selectedLocations

    def rmse(self, thisOutputs, target):
        loss = F.mse_loss(thisOutputs, target)
        loss = torch.sqrt(loss)

        return loss

    def perturb(self, inputs, target, clampEpsilon, stepNum):
        device = inputs.device

        if clampEpsilon == float('inf'):
            self.lr = 0.05
        else:
            self.lr = clampEpsilon * 2 / stepNum

        Xt = inputs.detach().clone().to(device).requires_grad_(True)
        optimizer = Adam([Xt], lr = self.lr, eps=self.epsilon)

        bestXt = inputs.detach().clone()
        bestForecast = torch.full(Xt.shape, float('inf')).to(device)
        loss_history = []

        for epoch in range(stepNum):
            optimizer.zero_grad()

            forecast = self.forward_func(Xt)
            loss = self.rmse(forecast, target)
            print(f"Epoch {epoch}, Loss: {loss.item()}")
            loss_history.append(loss.item())

            loss.backward()

            optimizer.step()

            with torch.no_grad():
                lower_bound = torch.maximum(inputs - clampEpsilon, torch.tensor(min_std, device=inputs.device))
                upper_bound = torch.minimum(inputs + clampEpsilon, torch.tensor(max_std, device=inputs.device))
                Xt.copy_(torch.clamp(Xt, lower_bound, upper_bound))

            with torch.no_grad():
                current_forecast = self.forward_func(Xt)

            if self.rmse(current_forecast, target).item() < self.rmse(bestForecast, target).item() and epoch > 0:
                bestXt = Xt.clone()
                bestForecast = current_forecast

            del forecast
            del loss

        torch.cuda.empty_cache()
        gc.collect()

        return bestXt, loss_history

class TAAOWPF:
    def __init__(self, forward_func, selectedLocations):
        super().__init__()
        self.forward_func = forward_func
        self.selectedLocations = selectedLocations

    def rmse(self, thisOutputs, target):
        loss = F.mse_loss(thisOutputs, target)
        loss = torch.sqrt(loss)

        return loss

    def perturb(self, inputs, target, clampEpsilon, stepNum):
        self.forward_func.eval()
        device = inputs.device

        if clampEpsilon == float('inf'):
            alpha = 0.05
        else:
            alpha = clampEpsilon * 2 / stepNum

        perturbedInputs = inputs.detach().clone().requires_grad_(True)
        optimizer = optim.Adam([perturbedInputs], lr = alpha)

        lossBest = float('inf')
        bestForecast = torch.full(inputs.shape, float('inf')).to(device)
        perturbedInputsBest = torch.empty_like(inputs)
        loss_history = []

        for epoch in range(stepNum):
            optimizer.zero_grad()
            thisOutputs = self.forward_func(perturbedInputs)
            loss = self.rmse(thisOutputs, target)
            print(f"Epoch {epoch}, Loss: {loss.item()}")
            loss_history.append(loss.item())
            loss.backward()

            perturbedInputs.grad.data = torch.sign(perturbedInputs.grad.data)
            optimizer.step()

            with torch.no_grad():
                lower_bound = torch.maximum(inputs - clampEpsilon, torch.tensor(min_std, device=inputs.device))
                upper_bound = torch.minimum(inputs + clampEpsilon, torch.tensor(max_std, device=inputs.device))
                perturbedInputs.copy_(torch.clamp(perturbedInputs, lower_bound, upper_bound))

            with torch.no_grad():
                currentForecast = self.forward_func(perturbedInputs)
            if self.rmse(currentForecast, target).item() < self.rmse(bestForecast, target).item() and epoch > 0:
                perturbedInputsBest = perturbedInputs.clone()
                bestForecast = currentForecast

            del thisOutputs
            del loss

        torch.cuda.empty_cache()
        gc.collect()

        return perturbedInputsBest, loss_history

class FGSM:
    def __init__(self, forward_func, selectedLocations):
        super().__init__()
        self.forward_func = forward_func
        self.selectedLocations = selectedLocations

    def rmse(self, thisOutputs, target):
        loss = F.mse_loss(thisOutputs, target)
        loss = torch.sqrt(loss)

        return loss

    def perturb(self, inputs, target, clampEpsilon, stepNum):
        self.forward_func.eval()
        device = inputs.device

        alpha = 0.1

        perturbedInputs = inputs.detach().clone().requires_grad_(True)

        outputs = self.forward_func(perturbedInputs)
        loss = self.rmse(outputs, target)
        loss.backward()
        gradSign = perturbedInputs.grad.data.sign()

        perturbedInputs = perturbedInputs - alpha * gradSign

        with torch.no_grad():
            lower_bound = torch.tensor(min_std, device=inputs.device)
            upper_bound = torch.tensor(max_std, device=inputs.device)
            perturbedInputs.copy_(torch.clamp(perturbedInputs, lower_bound, upper_bound))

        with torch.no_grad():
            currentForecast = self.forward_func(perturbedInputs)
        loss_history = [self.rmse(currentForecast, target).item()]

        torch.cuda.empty_cache()
        gc.collect()

        return perturbedInputs, loss_history

class STPGD:
    def __init__(self, model, selectedLocations, k, findType):
        self.model = model
        self.selectedLocations = selectedLocations
        self.k = k
        self.findType = findType

    def rmse(self, thisOutputs, target):
        loss = F.mse_loss(thisOutputs, target)
        loss = torch.sqrt(loss)
        return loss

    def perturb(self, inputs, target, clampEpsilon, stepNum):
        self.model.eval()
        device = inputs.device

        index = self.lookForSalientNodes(inputs, target, clampEpsilon, stepNum)

        chosenAttackNodes = torch.zeros_like(inputs)
        for batch in range(inputs.shape[0]):
            for idx in index[batch]:
                h, w = divmod(idx, 64)
                chosenAttackNodes[batch, :, h, w] = 1

        xPgd = inputs.detach().clone().requires_grad_(True)
        optimizer = optim.Adam([xPgd], lr= clampEpsilon * 2 / stepNum)

        lossBest = float("inf")
        xPgdBest = torch.empty_like(xPgd)
        bestForecast = torch.full(inputs.shape, float("inf")).to(device)
        loss_history = []

        for epoch in range(stepNum):
            optimizer.zero_grad()
            thisOutputs = self.model(xPgd)
            loss = self.rmse(thisOutputs, target)
            print(f"Epoch {epoch}, Loss: {loss.item()}")
            loss_history.append(loss.item())
            loss.backward()

            xPgd.grad.data = torch.sign(xPgd.grad.data) * chosenAttackNodes
            optimizer.step()

            with torch.no_grad():
                lower_bound = torch.maximum(inputs - clampEpsilon, torch.tensor(min_std, device=inputs.device))
                upper_bound = torch.minimum(inputs + clampEpsilon, torch.tensor(max_std, device=inputs.device))
                xPgd.copy_(torch.clamp(xPgd, lower_bound, upper_bound))

            with torch.no_grad():
                currentForecast = self.model(xPgd)

            if self.rmse(currentForecast, target).item() < self.rmse(bestForecast, target).item() and epoch > 0:
                xPgdBest = xPgd.clone()
                bestForecast = currentForecast

            del thisOutputs
            del loss

        torch.cuda.empty_cache()
        gc.collect()

        return xPgdBest, loss_history

    def lookForSalientNodes(self, X, target, clampEpsilon, stepNum):
        batch_size, time_steps, H, W = X.size()
        self.model.eval()

        if self.findType == "random":
            index = [random.sample(range(H * W), self.k) for _ in range(batch_size)]

        elif self.findType == "saliency":
            xSaliency = X.detach().clone().requires_grad_(True)

            saliencySteps = 1
            salientStepSize = clampEpsilon * 2 / stepStep if 'stepStep' in globals() else clampEpsilon * 2 / stepNum
            optimizer = optim.Adam([xSaliency], lr=salientStepSize)

            for epoch in range(saliencySteps):
                mseLoss = torch.nn.MSELoss()
                optimizer.zero_grad()
                thisOutputs = self.model(xSaliency)
                lossSaliency = mseLoss(thisOutputs, target)
                lossSaliency.backward()

                inputsGrad = xSaliency.grad.data

                del thisOutputs
                del lossSaliency
                del mseLoss
                torch.cuda.empty_cache()
                gc.collect()

            torch.cuda.empty_cache()
            index = self.attackSetBySaliencyMap(inputsGrad, self.k)

        return index

    def attackSetBySaliencyMap(self, inputGrads, k):
        chosenNodes = []
        for batch in range(inputGrads.shape[0]):
            thisInputGrads = inputGrads[batch].mean(dim=0)
            nodeSaliencyMap = []
            for h in range(thisInputGrads.shape[0]):
                for w in range(thisInputGrads.shape[1]):
                    nodeSaliency = torch.norm(F.relu(thisInputGrads[h, w])).item()
                    nodeSaliencyMap.append((nodeSaliency, h, w))

            nodeSaliencyMap.sort(reverse=True, key=lambda x: x[0])
            selected_indices = [h * 64 + w for _, h, w in nodeSaliencyMap[:k]]
            chosenNodes.append(selected_indices)

        return chosenNodes

def haar_3d_inverse(LLL, LLH, LHL, LHH, HLL, HLH, HHL, HHH):
    """
    Single-level 3D Haar inverse transform.
    Each sub-band has shape (B, T2, H2, W2).
    The output will have shape (B, T, H, W),
    where T=2*T2, H=2*H2, W=2*W2.
    """
    B, T2, H2, W2 = LLL.shape
    T = T2 * 2
    H = H2 * 2
    W = W2 * 2

    inv_sqrt8 = 1.0 / math.sqrt(8.0)
    x_recon = torch.zeros((B, T, H, W), device=LLL.device, dtype=LLL.dtype)

    x_recon[:, ::2, ::2, ::2]   = (LLL + LLH + LHL + LHH + HLL + HLH + HHL + HHH) * inv_sqrt8
    x_recon[:, ::2, ::2, 1::2]  = (LLL - LLH + LHL - LHH + HLL - HLH + HHL - HHH) * inv_sqrt8
    x_recon[:, ::2, 1::2, ::2]  = (LLL + LLH - LHL - LHH + HLL + HLH - HHL - HHH) * inv_sqrt8
    x_recon[:, ::2, 1::2, 1::2] = (LLL - LLH - LHL + LHH + HLL - HLH - HHL + HHH) * inv_sqrt8

    x_recon[:, 1::2, ::2, ::2]  = (LLL + LLH + LHL + LHH - HLL - HLH - HHL - HHH) * inv_sqrt8
    x_recon[:, 1::2, ::2, 1::2] = (LLL - LLH + LHL - LHH - HLL + HLH - HHL + HHH) * inv_sqrt8
    x_recon[:, 1::2, 1::2, ::2] = (LLL + LLH - LHL - LHH - HLL - HLH + HHL + HHH) * inv_sqrt8
    x_recon[:, 1::2, 1::2, 1::2] = (LLL - LLH - LHL + LHH - HLL + HLH + HHL - HHH) * inv_sqrt8

    # x_recon.shape: torch.Size([32, 8, 28, 58])

    return x_recon

def wavelet_decompose_3d_pywt(x_4d):
    """
    Single-level 3D wavelet transform with pywt.dwtn along (T,H,W).
    x_4d shape: (B, T, H, W).
    Returns dict of 8 sub-bands: 'LLL','LLH','LHL','LHH','HLL','HLH','HHL','HHH'.
    Each sub-band has shape (B, T/2, H/2, W/2).
    """
    x_np = x_4d.detach().cpu().numpy()                                          # Convert to numpy if needed
    # PyWavelets forward 3D transform
    coeffs_dict = pywt.dwtn(x_np, wavelet='haar', axes=(1,2,3))
    # Convert back to torch
    LLL = torch.tensor(coeffs_dict['aaa'], device=x_4d.device, dtype=x_4d.dtype)
    LLH = torch.tensor(coeffs_dict['aad'], device=x_4d.device, dtype=x_4d.dtype)
    LHL = torch.tensor(coeffs_dict['ada'], device=x_4d.device, dtype=x_4d.dtype)
    LHH = torch.tensor(coeffs_dict['add'], device=x_4d.device, dtype=x_4d.dtype)
    HLL = torch.tensor(coeffs_dict['daa'], device=x_4d.device, dtype=x_4d.dtype)
    HLH = torch.tensor(coeffs_dict['dad'], device=x_4d.device, dtype=x_4d.dtype)
    HHL = torch.tensor(coeffs_dict['dda'], device=x_4d.device, dtype=x_4d.dtype)
    HHH = torch.tensor(coeffs_dict['ddd'], device=x_4d.device, dtype=x_4d.dtype)

    return {'LLL': LLL, 'LLH': LLH, 'LHL': LHL, 'LHH': LHH, 'HLL': HLL, 'HLH': HLH, 'HHL': HHL, 'HHH': HHH}

class FABLE:
    def __init__(self, forward_func, wavelet, level, lons, lats):
        self.forward_func = forward_func
        self.wavelet = wavelet
        self.level = level
        self.lons = lons
        self.lats = lats
        self.lr = lr

    def rmse(self, pred, target):
        loss = F.mse_loss(pred, target)
        return torch.sqrt(loss)

    def wavelet_decompose_3d(self, data_4d):
        return wavelet_decompose_3d_pywt(data_4d)

    def wavelet_reconstruct_3d(self, coeffs_dict):
        return haar_3d_inverse(
            coeffs_dict['LLL'], coeffs_dict['LLH'], coeffs_dict['LHL'], coeffs_dict['LHH'],
            coeffs_dict['HLL'], coeffs_dict['HLH'], coeffs_dict['HHL'], coeffs_dict['HHH']
        )

    def create_regular_padded(self, lons, lats, values, fill_value=0.0):
        unique_lons = sorted(list(set(lons)))
        unique_lats = sorted(list(set(lats)))
        W = len(unique_lons)
        H = len(unique_lats)

        padded_data = np.full((H, W), fill_value, dtype=np.float32)
        lon_to_x = {val: idx for idx, val in enumerate(unique_lons)}
        lat_to_y = {val: idx for idx, val in enumerate(unique_lats)}

        for lon, lat, val in zip(lons, lats, values):
            ix = lon_to_x[lon]
            iy = lat_to_y[lat]
            padded_data[iy, ix] = val

        return padded_data, unique_lons, unique_lats

    def clamp_data(self, perturbed_data, original_data, clampEpsilon, mean, std):
        lower_bound = torch.maximum(original_data - clampEpsilon, torch.tensor(min_std, device=device))
        upper_bound = torch.minimum(original_data + clampEpsilon, torch.tensor(max_std, device=device))

        return torch.clamp(perturbed_data, lower_bound, upper_bound)

    def calculate_regularization_loss(self, perturbed_param_vector, original_param_vector, coeff_weights):
        reg_loss = 0
        key_to_index = {
            'LLL': 0, 'LLH': 1, 'LHL': 2, 'LHH': 3,
            'HLL': 4, 'HLH': 5, 'HHL': 6, 'HHH': 7
        }
        for key, weight in coeff_weights.items():
            index = key_to_index[key]
            diff = perturbed_param_vector[:, index, :, :, :] - original_param_vector[:, index, :, :, :]
            reg_loss += weight * torch.norm(diff, p=reg_p)
        return reg_loss

    def _forward_loss_given_params(
        self, param_vector,
        original_inputs,
        x_indices, y_indices,
        batch_size, seq_length,
        clampEpsilon, mean, std,
        target_torch, device
    ):
        """
        param_vector shape: (B, 8, T/2, H/2, W/2).
        Reconstruct with manual 3D inverse Haar.
        Map (B, T, H, W) -> (T, B, locations, 1), clamp, then compute loss.
        """
        # Build a dict of sub-bands
        coeffs_dict = {
            'LLL': param_vector[:, 0, :, :, :],
            'LLH': param_vector[:, 1, :, :, :],
            'LHL': param_vector[:, 2, :, :, :],
            'LHH': param_vector[:, 3, :, :, :],
            'HLL': param_vector[:, 4, :, :, :],
            'HLH': param_vector[:, 5, :, :, :],
            'HHL': param_vector[:, 6, :, :, :],
            'HHH': param_vector[:, 7, :, :, :],
        }

        reconstructed_4d = self.wavelet_reconstruct_3d(coeffs_dict)             # Reconstruct: shape (B, T, H, W)
        recon_perm = reconstructed_4d.permute(1, 0, 2, 3)                       # Permute to (T, B, H, W)

        # Flatten so we can gather
        width = recon_perm.shape[-1]                                            # recon_perm: (T,B,H,W)
        recon_2d = recon_perm.reshape(seq_length, batch_size, -1)               # shape => (T,B,H*W)

        # Gather only the valid indices
        linear_idx = y_indices * width + x_indices
        perturbed_inputs_2d = torch.gather(
            recon_2d, dim=2,
            index=linear_idx.unsqueeze(0).unsqueeze(0).expand(seq_length, batch_size, -1)
        )                                                                       # (T,B,locations)
        perturbed_inputs_4d = perturbed_inputs_2d.unsqueeze(-1)                 # (T,B,locations,1)

        # Clamp
        print(perturbed_inputs_4d.shape, original_inputs.shape)
        perturbed_inputs_4d = self.clamp_data(perturbed_inputs_4d, original_inputs, clampEpsilon, mean, std)

        # Forward pass
        forecast = self.forward_func(perturbed_inputs_4d.to(device))
        loss_val = self.rmse(forecast, target_torch)
        # print(forecast.shape, target_torch.shape)
        loss_val = self.rmse(forecast, target_torch)
        return perturbed_inputs_4d, loss_val

    def perturb(self, inputs, target, clampEpsilon, step_num, mean, std):
        """
        inputs: (T,B,locations,1)
        target: (T,B,locations,1)
        """
        device = inputs.device
        self.forward_func.eval()

        seq_length, batch_size, locations, _ = inputs.shape

        # Example: record lon/lat for each time step in each batch
        ulonlat_record = []
        padded_data_batch = []

        # Build a (B,T,H,W) data array
        for b_idx in range(batch_size):
            time_step_padded = []
            time_step_ulonlat = []
            for t in range(seq_length):
                data_1d = inputs[t, b_idx, :, 0].detach().cpu().numpy()
                padded_data, unique_lons, unique_lats = self.create_regular_padded(
                    self.lons, self.lats, data_1d, fill_value=0.0
                )
                time_step_padded.append(padded_data)
                time_step_ulonlat.append((unique_lons, unique_lats))
            padded_data_batch.append(np.stack(time_step_padded, axis=0))
            ulonlat_record.append(time_step_ulonlat)

        padded_data_batch = np.stack(padded_data_batch, axis=0)  # (B, T, H, W)
        padded_data_batch_torch = torch.tensor(padded_data_batch, dtype=torch.float32, device=device)

        # Record the valid indices in the tensor 'padded_data_batch'
        unique_lons, unique_lats = ulonlat_record[0][0]  # use the first sample's mapping
        lon_to_x = {val: idx for idx, val in enumerate(unique_lons)}
        lat_to_y = {val: idx for idx, val in enumerate(unique_lats)}
        valid_indices = [
            (lon_to_x[lon], lat_to_y[lat])
            for lon, lat in zip(self.lons, self.lats)
            if lon in lon_to_x and lat in lat_to_y
        ]
        x_indices, y_indices = zip(*valid_indices)
        x_indices = torch.tensor(x_indices, dtype=torch.long, device=device)
        y_indices = torch.tensor(y_indices, dtype=torch.long, device=device)

        # 3D forward wavelet transform with pywt
        coeffs = self.wavelet_decompose_3d(padded_data_batch_torch)
        # Build param_vector => shape (B, 8, T/2, H/2, W/2)
        param_vector = torch.stack([
            coeffs['LLL'], coeffs['LLH'], coeffs['LHL'], coeffs['LHH'],
            coeffs['HLL'], coeffs['HLH'], coeffs['HHL'], coeffs['HHH']
        ], dim=1)

        key_to_index = {'LLL': 0, 'LLH': 1, 'LHL': 2, 'LHH': 3, 'HLL': 4, 'HLH': 5, 'HHL': 6, 'HHH': 7}
        # Suppose we only want to attack LLL, LHH => for example
        target_indices = [key_to_index[k] for k in target_key]
        non_target_indices = [i for i in range(8) if i not in target_indices]

        # Keep a copy of the original sub-bands for the non-target ones
        unchanged_param = param_vector[:, non_target_indices, :, :, :].detach().clone()

        perturbed_param_vector = param_vector.detach().clone()
        perturbed_param_vector.requires_grad_(True)

        target_torch = target.clone().detach().to(device)

        # For example, use optimizer = 'Adam'
        if use_optimizer == 'Adam':
            optimizer = torch.optim.Adam([perturbed_param_vector], lr=self.lr)

        best_loss = float('inf')
        best_perturbed_inputs = None

        loss_history = []

        for epoch in range(step_num):
            optimizer.zero_grad()
            perturbed_inputs, loss = self._forward_loss_given_params(
                perturbed_param_vector,
                inputs,
                x_indices, y_indices,
                batch_size=batch_size,
                seq_length=seq_length,
                clampEpsilon=clampEpsilon,
                mean=mean,
                std=std,
                target_torch=target_torch,
                device=device
            )

            ### regularizer
            if reg == 1:
                reg_loss = self.calculate_regularization_loss(perturbed_param_vector, param_vector, coeff_weights)
                total_loss = loss + lambda_reg * reg_loss
                print(f"Epoch {epoch} | Total Loss: {total_loss.item():.6f}, Loss: {loss.item():.6f}, reg_loss: {reg_loss:.6f}")
                total_loss.backward()
            else:
                print(f'Epoch {epoch}', ' | Loss:', loss.item(), ' | GradNorm:', torch.norm(perturbed_param_vector.grad.data).item())
                loss.backward()

            loss_history.append(loss.item())

            # Zero out grad for non-target sub-bands
            if perturbed_param_vector.grad is not None:
                perturbed_param_vector.grad[:, non_target_indices, :, :, :] = 0

            optimizer.step()

            # Restore non-target sub-bands
            with torch.no_grad():
                perturbed_param_vector[:, non_target_indices, :, :, :] = unchanged_param

            if loss.item() < best_loss:
                best_loss = loss.item()
                best_perturbed_inputs = perturbed_inputs

        return best_perturbed_inputs, loss_history

In [ ]:
# @title Implement Adversarial Attack

if dataset == "ERA5":
    test_X = torch.load(test_x_path)                                            # Shape: [60, 12, 32, 64]
    test_Y = torch.load(test_y_path)                                            # Shape: [60, 12, 32, 64]
    print(f"Test X shape: {test_X.shape}, Test Y shape: {test_Y.shape}")
    test_X = (test_X - mean) / std
    test_Y = (test_Y - mean) / std
    test_dataset = torch.utils.data.TensorDataset(test_X, test_Y)
    test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
elif dataset == "NLDAS":
    test_loader = clcrnModelTrained._data['{}_loader'.format('test')]

if baseline == 'taaowpf':
    taaowpf = TAAOWPF(model, selectedLocations)
elif baseline == 'noise_attack':
    noiseAttack = NoiseAttack(model, selectedLocations)
elif baseline == 'stpgd':
    stpgd = STPGD(model, selectedLocations, k, findType)
elif baseline == 'ala':
    ala = ALA(model, selectedLocations)
elif baseline == "fgsm":
    fgsm = FGSM(model, selectedLocations)
elif baseline == "fable":
    fable = FABLE(forward_func=model, wavelet='haar', level=1, lons=lons, lats=lats)

breakFlag = 0

for run in range(num_runs):
    data_list = []
    run_loss_list = []
    for loc in range(num_locs):
        print(f'run {run} and loc {loc}')
        predList = torch.load(os.path.join(save_dir_base, f'predList_{dataset}_{dataName}_{loc}_{run}.pth'))
        targetList = torch.load(os.path.join(save_dir_base, f'targetList_{dataset}_{dataName}_{loc}_{run}.pth'))

        # print(len(predList))

        for batch_idx, (x, y) in enumerate(test_loader):
            if x.shape[0] != batch_size:
                break
            print(f"Processing batch {batch_idx}, with x ({x.shape}) and y ({y.shape})")

            thisX = x.to('cuda')                                                    # Shape: [batch_size, 12, 32, 64]
            yPred = predList[batch_idx].to('cuda')
            yTarget = targetList[batch_idx].to('cuda')

            if dlwfm == "CLCRN":
                thisX = x.permute(1, 0, 2, 3).to('cuda')
                yPred = predList[batch_idx].permute(1, 0, 2, 3).to('cuda')
                yTarget = targetList[batch_idx].permute(1, 0, 2, 3).to('cuda')

            torch.cuda.reset_peak_memory_stats()
            start_time = time.time()

            if baseline == 'taaowpf':
                xPertubed, loss_history = taaowpf.perturb(thisX, yTarget, clampEpsilon, stepNum)
            elif baseline == 'noise_attack':
                xPertubed, loss_history = noiseAttack.perturb(thisX, yTarget, clampEpsilon, stepNum)
            elif baseline == 'stpgd':
                xPertubed, loss_history = stpgd.perturb(thisX, yTarget, clampEpsilon, stepNum)
            elif baseline == 'ala':
                xPertubed, loss_history = ala.perturb(thisX, yTarget, clampEpsilon, stepNum)
            elif baseline == "fgsm":
                xPertubed, loss_history = fgsm.perturb(thisX, yTarget, clampEpsilon, stepNum)
            elif baseline == "fable":
                xPertubed, loss_history = fable.perturb(thisX, yTarget, clampEpsilon, stepNum, mean, std)

            end_time = time.time()

            elapsed_time = end_time - start_time
            peak_mem = torch.cuda.max_memory_allocated() / 1024**2              # in MB
            print(elapsed_time, peak_mem)

            yPerturbed = model(xPertubed.to('cuda'))

            # print(f"thisX: {thisX.shape}, yPred: {yPred.shape}, yTarget: {yTarget.shape}, xPertubed: {xPertubed.shape}, yPerturbed: {yPerturbed.shape}")

            loss_value = torch.sqrt(F.mse_loss(yPerturbed, yTarget)).item()
            run_loss_list.append({
                "run": run,
                "loc": loc,
                "batch": batch_idx,
                "loss": loss_value,
                "time": elapsed_time,
                "memory_MB": peak_mem
            })

            for batch in range(x.shape[0]):  # Loop over samples in batch
                if dlwfm == "FourCastNet":
                    data_list.append({
                        "thisX": thisX[batch].reshape(12, -1, 1).detach().cpu().numpy().tolist(),  # [12, 2048, 1]
                        "yPred": yPred[batch].reshape(12, -1, 1).detach().cpu().numpy().tolist(),  # [12, 2048, 1]
                        "yTarget": yTarget[batch].reshape(12, -1, 1).detach().cpu().numpy().tolist(),  # [12, 2048, 1]
                        "xPerturbed": xPertubed[batch].reshape(12, -1, 1).detach().cpu().numpy().tolist(),  # [12, 2048, 1]
                        "yPerturbed": yPerturbed[batch].reshape(12, -1, 1).detach().cpu().numpy().tolist(),  # [12, 2048, 1]
                    })
                elif dlwfm == "CLCRN":
                    data_list.append({
                        "thisX": thisX[:, batch, :, :].detach().cpu().numpy().tolist(),
                        "yPred": yPred[:, batch, :, :].detach().cpu().numpy().tolist(),
                        "yTarget": yTarget[:, batch, :, :].detach().cpu().numpy().tolist(),
                        "xPerturbed": xPertubed[:, batch, :, :].detach().cpu().numpy().tolist(),
                        "yPerturbed": yPerturbed[:, batch, :, :].detach().cpu().numpy().tolist(),
                    })

            loss_df = pd.DataFrame({
                "run": [run]*len(loss_history),
                "loc": [loc]*len(loss_history),
                "batch": [batch_idx]*len(loss_history),
                "epoch": list(range(len(loss_history))),
                "loss": loss_history
            })
            if baseline != "fable":
                loss_path = os.path.join(out_path_base, "attack_loss", f"loss_run{run}_loc{loc}_batch{batch_idx}_{stepNum}_{clampEpsilon}.csv")
            elif baseline == "fable":
                loss_path = os.path.join(out_path_base, "attack_loss", f"loss_run{run}_loc{loc}_batch{batch_idx}_{stepNum}_{clampEpsilon}_{lr}_{lambda_reg}_{reg_p}.csv")
            loss_df.to_csv(loss_path, index=False)

            del thisX, yPred, yTarget, xPertubed, yPerturbed
            torch.cuda.empty_cache()
            gc.collect()

    res_df = pd.DataFrame(data_list)
    if baseline != "fable":
        out_res_path = os.path.join(out_path_base, f"out_run{run}_{stepNum}_{clampEpsilon}.csv")
    elif baseline == "fable":
        out_res_path = os.path.join(out_path_base, f"out_run{run}_{stepNum}_{clampEpsilon}_{lr}_{lambda_reg}_{reg_p}.csv")
    res_df.to_csv(out_res_path, index=False)

    run_loss_df = pd.DataFrame(run_loss_list)
    if baseline != "fable":
        run_loss_path = os.path.join(out_path_base, "loss_records", f"loss_run{run}_{stepNum}_{clampEpsilon}.csv")
    elif baseline == "fable":
        run_loss_path = os.path.join(out_path_base, "loss_records", f"loss_run{run}_{stepNum}_{clampEpsilon}_{lr}_{lambda_reg}_{reg_p}.csv")
    run_loss_df.to_csv(run_loss_path, index=False)

torch.cuda.empty_cache()
gc.collect()


run 0 and loc 0
Processing batch 0, with x (torch.Size([48, 12, 1320, 1])) and y (torch.Size([48, 12, 1320, 1]))
torch.Size([12, 48, 1320, 1]) torch.Size([12, 48, 1320, 1])
torch.Size([12, 48, 1320, 1]) torch.Size([12, 48, 1320, 1])
Epoch 0 | Total Loss: 0.005223, Loss: 0.005223, reg_loss: 0.000000
torch.Size([12, 48, 1320, 1]) torch.Size([12, 48, 1320, 1])
Epoch 1 | Total Loss: 0.034537, Loss: 0.030412, reg_loss: 41.256691
torch.Size([12, 48, 1320, 1]) torch.Size([12, 48, 1320, 1])
Epoch 2 | Total Loss: 0.051086, Loss: 0.043753, reg_loss: 73.325661
2.7518579959869385 8088.767578125
Processing batch 1, with x (torch.Size([48, 12, 1320, 1])) and y (torch.Size([48, 12, 1320, 1]))
torch.Size([12, 48, 1320, 1]) torch.Size([12, 48, 1320, 1])
torch.Size([12, 48, 1320, 1]) torch.Size([12, 48, 1320, 1])
Epoch 0 | Total Loss: 0.005203, Loss: 0.005203, reg_loss: 0.000000
torch.Size([12, 48, 1320, 1]) torch.Size([12, 48, 1320, 1])
Epoch 1 | Total Loss: 0.033703, Loss: 0.029583, reg_loss: 41.20215

0

In [ ]:
# @title Implement evaluation
dim = 0 # dim=0 (or 1 only for 'WeatherBench-surface wind components')

!pip install pysal esda libpysal

from google.colab import drive
import pandas as pd
import numpy as np
import os
import torch
import pickle
import xarray as xr

from libpysal.weights import DistanceBand
from scipy.spatial import distance_matrix
from esda.moran import Moran
import statsmodels.api as sm
from scipy.stats import zscore

def spatialMoranI(X, W):
        moran = Moran(X, W)
        return moran.I

def temporalAutocorrelation(timeSeries):
    acfValues = sm.tsa.acf(timeSeries)

    if not np.isnan(acfValues).any():
        return acfValues
    else:
        return np.zeros(len(acfValues))

summary_results = []
for run in range(num_runs):
    if baseline != "fable":
        out_res_path = os.path.join(out_path_base, f"out_run{run}_{stepNum}_{clampEpsilon}.csv")
    elif baseline == "fable":
        out_res_path = os.path.join(out_path_base, f"out_run{run}_{stepNum}_{clampEpsilon}_{lr}_{lambda_reg}_{reg_p}.csv")
    df = pd.read_csv(out_res_path)

    gPrecision = []
    inPrecision = []
    outPrecision = []
    closeness = []
    sMoranDiffAvgX = []
    tAutoDiffAvgX = []

    sMoranXListAll = []
    sMoranXAdvListAll = []

    # Store in dictionary
    lonLatDict = {"lonlat": lonlat}

    print(len(df['thisX']))
    for sampleNum in range(len(df['thisX'])): # a list of tensors
        print(f"run{run}, sampleNum{sampleNum}")

        # JUST USED FOR TESTING CODE CORRECTNESS
        # if sampleNum > 2: break

        ### in-target precision
        # format of 'inPositions': tensor([[timestep, location], [], ..., []])
        inPositions =  torch.nonzero(
            torch.tensor(eval(df['yTarget'][sampleNum]))[:, :, dim]
            - torch.tensor(eval(df['yPred'][sampleNum]))[:, :, dim])
        thisInPrecision = torch.sum(
            torch.abs(
                torch.tensor(eval(df['yTarget'][sampleNum]))[inPositions[:, 0], inPositions[:, 1], dim]
                - torch.tensor(eval(df['yPerturbed'][sampleNum]))[inPositions[:, 0], inPositions[:, 1], dim])
            )
        inPrecision.append(thisInPrecision.item())

        ### out-target precision.
        outPositions = torch.nonzero(
            (torch.tensor(eval(df['yTarget'][sampleNum]))[:, :, dim]
            - torch.tensor(eval(df['yPred'][sampleNum]))[:, :, dim])==0)
        thisOutPrecision = torch.sum(
            torch.abs(
                torch.tensor(eval(df['yTarget'][sampleNum]))[outPositions[:, 0], outPositions[:, 1], dim]
                - torch.tensor(eval(df['yPerturbed'][sampleNum]))[outPositions[:, 0], outPositions[:, 1], dim])
            )
        outPrecision.append(thisOutPrecision.item())

        ### closeness
        thisCloseness = torch.mean(
            torch.abs(torch.tensor(eval(df['thisX'][sampleNum]))[:, :, dim]
            - torch.tensor(eval(df['xPerturbed'][sampleNum]))[:, :, dim]))
        closeness.append(thisCloseness.item())

        ### spatial autocorrelation-X
        lonlat = lonLatDict['lonlat']
        threshold = 6                                                               # This is a hyperparameter

        W = DistanceBand(lonlat, threshold=threshold, binary=False, silence_warnings=True) # w.n: 1320*1320
        if np.all(W.s0 == 0):
            print("Warning: Empty weight matrix W, check threshold!")
        X = torch.tensor(eval(df['thisX'][sampleNum]))[:, :, dim].numpy()           # X.shape: 1320*7
        XAdv = torch.tensor(eval(df['xPerturbed'][sampleNum]))[:, :, dim].numpy()
        sMoranXList = [spatialMoranI(X[i,:].flatten(), W) for i in range(X.shape[0])] # for each element in the list, the shape is (1320,)
        sMoranXAdvList = [spatialMoranI(XAdv[i,:].flatten(), W) for i in range(XAdv.shape[0])]
        sMoranXDiffList = [abs(a - b) for a, b in zip(sMoranXList, sMoranXAdvList)]
        thisSMoranDiffAvgX = np.sum(sMoranXDiffList)
        sMoranDiffAvgX.append(thisSMoranDiffAvgX)
        sMoranXListAll.append(sMoranXList)
        sMoranXAdvListAll.append(sMoranXAdvList)

        ### temporal autocorrelation-X
        tAutoXList = [temporalAutocorrelation(X[:,i]) for i in range(X.shape[1])]   # For NLDAS-precipitation dataset, there are some n.a. outcomes for temporal autocorrelation. So, we first replace them as all zeros and then take those invalid out
        tAutoXNp = np.array(tAutoXList)                                             # len(tAutoXList): 1320. tAutoXNp.shape: (1320, 7)
        tAutoXAdvList = [temporalAutocorrelation(XAdv[:,i]) for i in range(XAdv.shape[1])]
        tAutoXAdvNp = np.array(tAutoXAdvList)
        tAutoDiffNp = np.abs(tAutoXNp - tAutoXAdvNp)                                # tAutoDiffNp.shape: (1320, 7)
        thisTAutoDiffAvgX = np.mean(tAutoDiffNp, axis=1)
        thisTAutoDiffAvgX = np.sum(thisTAutoDiffAvgX)
        tAutoDiffAvgX.append(thisTAutoDiffAvgX)

        print(thisInPrecision.item(), thisOutPrecision.item(), thisCloseness.item(), thisSMoranDiffAvgX, thisTAutoDiffAvgX)

    mean_in = np.mean(inPrecision)
    mean_out = np.mean(outPrecision)
    mean_close = np.mean(closeness)
    mean_sMoran = np.mean(sMoranDiffAvgX) / seq_length
    mean_tAuto = np.mean(tAutoDiffAvgX) / grid_size
    print(f"Run {run} Summary:")
    print(mean_in, mean_out, mean_close, mean_sMoran, mean_tAuto)

    dict = {
        'In-Target Precision': inPrecision,
        'Out-Target Precision': outPrecision,
        'Closeness': closeness,
        'Spatial Autocorrelation X': sMoranDiffAvgX,
        'Temporal Autocorrelation X': tAutoDiffAvgX,
        'Moran X': sMoranXListAll,
        'Moran XAdv': sMoranXAdvListAll
    }

    df = pd.DataFrame(dict)

    if baseline != "fable":
        eval_path = os.path.join(eval_path_base, f"eval_run{run}_{stepNum}_{clampEpsilon}.csv")
    elif baseline == "fable":
        eval_path = os.path.join(eval_path_base, f"eval_run{run}_{stepNum}_{clampEpsilon}_{lr}_{lambda_reg}_{reg_p}.csv")

    df.to_csv(eval_path, index = False)

    summary_results.append({
        "run": run,
        "In-Target Precision": mean_in,
        "Out-Target Precision": mean_out,
        "Closeness": mean_close,
        "Spatial Autocorrelation X": mean_sMoran,
        "Temporal Autocorrelation X": mean_tAuto
    })

summary_df = pd.DataFrame(summary_results)
summary_stats = summary_df.describe().loc[['mean', 'std']]
summary_final = pd.concat([summary_df, summary_stats])

if baseline != "fable":
    summary_path = os.path.join(eval_path_base, f"summary_all_runs_{stepNum}_{clampEpsilon}.csv")
else:
    summary_path = os.path.join(eval_path_base, f"summary_all_runs_{stepNum}_{clampEpsilon}_{lr}_{lambda_reg}_{reg_p}.csv")

summary_final.to_csv(summary_path, index=True)
print(f"Saved overall summary with mean/std to: {summary_path}")


960
run0, sampleNum0


/usr/local/lib/python3.12/dist-packages/scipy/sparse/_data.py:128: RuntimeWarning: divide by zero encountered in reciprocal
  return self._with_data(data ** n)


2.2699973583221436 0.0008348412811756134 2.9359339848156196e-08 5.48930782162671e-07 9.646591092794997e-05
run0, sampleNum1
2.298558235168457 0.0007943250238895416 3.471559395507029e-08 4.893767928670911e-07 8.813838200345274e-05
run0, sampleNum2
2.1621413230895996 0.0007800720632076263 2.7306713334951382e-08 3.846661810191776e-07 8.31365913462458e-05
run0, sampleNum3
Run 0 Summary:
2.243565638860067 0.0008030794560909271 3.046054904605929e-08 3.952704877913721e-08 6.76113344135476e-08
960
run1, sampleNum0


/usr/local/lib/python3.12/dist-packages/scipy/sparse/_data.py:128: RuntimeWarning: divide by zero encountered in reciprocal
  return self._with_data(data ** n)


2.171452045440674 0.0008348412811756134 2.9359339848156196e-08 5.48930782162671e-07 9.646591092794997e-05
run1, sampleNum1
2.255814790725708 0.0007943250238895416 3.471559395507029e-08 4.893767928670911e-07 8.813838200345274e-05
run1, sampleNum2
2.110116958618164 0.0007800720632076263 2.7306713334951382e-08 3.846661810191776e-07 8.31365913462458e-05
run1, sampleNum3
Run 1 Summary:
2.1791279315948486 0.0008030794560909271 3.046054904605929e-08 3.952704877913721e-08 6.76113344135476e-08
960
run2, sampleNum0


/usr/local/lib/python3.12/dist-packages/scipy/sparse/_data.py:128: RuntimeWarning: divide by zero encountered in reciprocal
  return self._with_data(data ** n)


2.2493016719818115 0.0008348412811756134 2.9359339848156196e-08 5.48930782162671e-07 9.646591092794997e-05
run2, sampleNum1
2.2805542945861816 0.0007943250238895416 3.471559395507029e-08 4.893767928670911e-07 8.813838200345274e-05
run2, sampleNum2
2.1305770874023438 0.0007800720632076263 2.7306713334951382e-08 3.846661810191776e-07 8.31365913462458e-05
run2, sampleNum3
Run 2 Summary:
2.220144351323446 0.0008030794560909271 3.046054904605929e-08 3.952704877913721e-08 6.76113344135476e-08
960
run3, sampleNum0


/usr/local/lib/python3.12/dist-packages/scipy/sparse/_data.py:128: RuntimeWarning: divide by zero encountered in reciprocal
  return self._with_data(data ** n)


KeyboardInterrupt: 

In [ ]:
# @title Construct adversarial target

import math
import torch
import numpy as np
import random
import os
from google.colab import drive

BASE_SEED = 12345
NUM_WORKERS = 4

random.seed(BASE_SEED)
np.random.seed(BASE_SEED)
torch.manual_seed(BASE_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(BASE_SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

def worker_init_fn(worker_id):
    worker_seed = BASE_SEED + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)

locations_list = [628, 17, 668, 117, 226, 830, 290, 547, 975, 122]              # [628, 17, 668, 117, 226, 830, 290, 547, 975, 122]
                                                                                # [New York, Miami, Chicago, Houston, Dallas, Minneapolis,  Los Angeles, Denver, Seattle, New Orleans]
if dataset == "ERA5":
    test_X = torch.load(test_x_path)                                            # Shape: [60, 12, 32, 64]
    test_Y = torch.load(test_y_path)                                            # Shape: [60, 12, 32, 64]
    print(f"Test X shape: {test_X.shape}, Test Y shape: {test_Y.shape}")
    test_X = (test_X - mean) / std
    test_Y = (test_Y - mean) / std
    test_dataset = torch.utils.data.TensorDataset(test_X, test_Y)
    test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
elif dataset == "NLDAS":
    test_loader = clcrnModelTrained._data['{}_loader'.format('test')]

for run in range(num_runs):
    for loc in range(num_locs):
        # Dataset statistics (standardized values)
        seed = BASE_SEED + run * 100 + loc
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)

        g = torch.Generator()
        g.manual_seed(seed)

        if dataset == "ERA5":
            test_loader = torch.utils.data.DataLoader(
                test_dataset,
                batch_size=batch_size if 'batch_size' in globals() else 1,
                shuffle=False,
                num_workers=NUM_WORKERS,
                worker_init_fn=worker_init_fn,
                pin_memory=True,
                generator=g
            )

        predList = []
        targetList = []

        for _, (x, y) in enumerate(test_loader):
            if x.shape[0] != batch_size:
                break

            with torch.no_grad():
                yPred = model(x.permute(1, 0, 2, 3).to('cuda'))
                yPred = yPred.permute(1, 0, 2, 3)

            # (batch_size, time_step, locations, variables)
            yTarget = yPred.clone()

            for batch in range(yPred.shape[0]):
                # location = torch.randint(0, yPred.shape[2] - 1, (1,)).item()
                location = locations_list[loc]
                center = random.choice([5, 6])
                sigma = 6

                if yPred[batch, center, location, 0] < p50:
                    target_center = yPred[batch, center, location, 0] - random.uniform(9 / std, 10 / std)
                else:
                    target_center = yPred[batch, center, location, 0] + random.uniform(9 / std, 10 / std)

                yTarget[batch, center, location, 0] = target_center
                delta = target_center - yPred[batch, center, location, 0]

                for t in range(12):
                    if t == center:
                        continue
                    factor = np.exp(-((t - center) ** 2) / sigma**2)
                    yTarget[batch, t, location, 0] = yPred[batch, t, location, 0] + delta * factor

                # yTarget[batch, 0, location, 0] = yPred[batch, 0, location, 0]
                # yTarget[batch, 11, location, 0] = yPred[batch, 11, location, 0]

                yTarget[batch, :, location, :] = torch.clamp(yTarget[batch, :, location, :], min=min_std, max=max_std)

            predList.append(yPred.cpu())
            targetList.append(yTarget.cpu())

        torch.save(predList, os.path.join(save_dir_base, f'predList_{dataset}_{dataName}_{loc}_{run}.pth'))
        torch.save(targetList, os.path.join(save_dir_base, f'targetList_{dataset}_{dataName}_{loc}_{run}.pth'))
